In [1]:
pip install sqlalchemy pyodbc pandas

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyodbc]emy]
Note: you may need to restart the kernel to use updated packages.


### download ODBC driver18

brew tap microsoft/mssql-release https://github.com/Microsoft/homebrew-mssql-release

brew update

brew install unixodbc

ACCEPT_EULA=Y brew install msodbcsql18

python -c "import pyodbc; print(pyodbc.drivers())"

In [ ]:
import os
import math
import pandas as pd
from sqlalchemy import create_engine, text
import urllib.parse
import numpy as np

# ===============================
# Azure SQL credentials
# ===============================
SERVER = "..."
DB     = "..."
USER   = "..."
PWD    = "..." 
DRIVER = "ODBC Driver 18 for SQL Server"

SCHEMA = "dw"

# ===============================
# CSV directory (change if needed)
# ===============================
EXPORT_DIR = "export_mailchimp/20260219T215720Z"  # <-- change to your folder

# If you want to load from the uploaded sandbox files instead, set:
# EXPORT_DIR = "/mnt/data"

FILES = [
    ("dim_user", "dim_user.csv"),
    ("dim_date", "dim_date.csv"),
    ("dim_mailchimp_audience", "dim_mailchimp_audience.csv"),
    ("dim_mailchimp_campaign", "dim_mailchimp_campaign.csv"),
    ("fact_mailchimp_audience_monthly", "fact_mailchimp_audience_monthly.csv"),
    ("fact_mailchimp_campaign_monthly", "fact_mailchimp_campaign_monthly.csv"),
    ("fact_mailchimp_campaign_rank_monthly", "fact_mailchimp_campaign_rank_monthly.csv"),
]

# ===============================
# Create engine (same style as yours)
# ===============================
conn_str = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DB};"
    f"UID={USER};"
    f"PWD={PWD};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(conn_str),
    fast_executemany=True,
)

# -------------------------------
# Cleaning helpers
# -------------------------------
DATE_COLS = {"date", "week_start_date", "week_end_date", "month_start_date"}
BIT_COLS  = {"is_weekend", "is_holiday"}

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # 1) Convert known date cols to python date
    for col in (DATE_COLS & set(df.columns)):
        df[col] = pd.to_datetime(df[col], errors="coerce").dt.date

    # 2) Convert BIT cols to 0/1/None
    for col in (BIT_COLS & set(df.columns)):
        df[col] = df[col].map(
            lambda x: None if pd.isna(x) else (1 if str(x).strip().lower() in {"1","true","t","yes","y"} else 0)
        )

    # 3) Force object dtype so NULLs become real Python None (prevents float NaN staying NaN)
    df = df.astype("object")

    # 4) Replace ALL NaN/NaT/NA (including float NaN) with None
    df = df.where(pd.notnull(df), None)

    # 5) Extra safety: convert any remaining numpy.nan to None
    df = df.replace({np.nan: None})

    return df

def quoted_insert_statement(schema: str, table: str, cols: list[str]):
    col_sql = ", ".join(f"[{c}]" for c in cols)
    val_sql = ", ".join(f":{c}" for c in cols)
    return text(f"INSERT INTO {schema}.{table} ({col_sql}) VALUES ({val_sql})")

def delete_all(schema: str, table: str):
    with engine.begin() as conn:
        conn.execute(text(f"DELETE FROM {schema}.{table};"))

def load_table(schema: str, table: str, csv_path: str, batch_size: int = 300, wipe: bool = True):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    df = clean_df(df)

    if wipe:
        delete_all(schema, table)

    cols = list(df.columns)
    stmt = quoted_insert_statement(schema, table, cols)

    records = df.to_dict(orient="records")
    total = len(records)
    if total == 0:
        print(f"⚠️  {schema}.{table}: empty CSV, skipped.")
        return

    with engine.begin() as conn:
        for i in range(0, total, batch_size):
            conn.execute(stmt, records[i:i+batch_size])

    print(f"✅ Loaded {schema}.{table}: {total} rows")

for table, filename in FILES:
    path = os.path.join(EXPORT_DIR, filename)
    print(f"\nLoading {filename} -> {SCHEMA}.{table}")
    load_table(SCHEMA, table, path, batch_size=300, wipe=True)



Loading dim_user.csv -> dw.dim_user
✅ Loaded dw.dim_user: 1 rows

Loading dim_date.csv -> dw.dim_date
✅ Loaded dw.dim_date: 415 rows

Loading dim_mailchimp_audience.csv -> dw.dim_mailchimp_audience
✅ Loaded dw.dim_mailchimp_audience: 21 rows

Loading dim_mailchimp_campaign.csv -> dw.dim_mailchimp_campaign
✅ Loaded dw.dim_mailchimp_campaign: 17 rows

Loading fact_mailchimp_audience_monthly.csv -> dw.fact_mailchimp_audience_monthly
✅ Loaded dw.fact_mailchimp_audience_monthly: 56 rows

Loading fact_mailchimp_campaign_monthly.csv -> dw.fact_mailchimp_campaign_monthly
✅ Loaded dw.fact_mailchimp_campaign_monthly: 17 rows

Loading fact_mailchimp_campaign_rank_monthly.csv -> dw.fact_mailchimp_campaign_rank_monthly
✅ Loaded dw.fact_mailchimp_campaign_rank_monthly: 102 rows
